In [ ]:
# Test-Dataset Experiment with BAliBASE BB11001


# parse data from FASTA

from pathlib import Path

# path to dataset and read data lines
fasta_path = Path("datasets/BAliBASE/inputs/BB11001.fasta")   # check your path!!!!

with open(fasta_path, "r") as f:
    lines = f.readlines()

print(lines[:5])

# prepare
proteins = {}
protein_name = None
sequence_parts = []

for line in lines:
    line = line.strip()
    if not line:
        continue #skip

    if line.startswith(">"):
        if protein_name is not None:
            proteins[protein_name] = "".join(sequence_parts)

        protein_name = line.removeprefix(">")
        sequence_parts = []
        
    else:
        sequence_parts.append(line)

# last protein after EOF
if protein_name is not None:
    proteins[protein_name] = "".join(sequence_parts)


# display proteins
print("Number of proteins:", len(proteins))

for name, sequence in proteins.items():
    print(name, ":", len(sequence), "residues")


In [ ]:
# prepare token-IDs use Hugging-Face-Tokenizer for ESM-2

from transformers import AutoTokenizer

# select protein
protein_name ="1aab_"
protein_sequence = proteins[protein_name]

# model
model_name = "facebook/esm2_t33_650M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokens = tokenizer(protein_sequence, return_tensors="pt")

# start-token: 0 = <cls>, end-token: 2 = <eos>

# results
print("Protein:", protein_name)
print("Amino acid sequence length:", len(protein_sequence))
print("Token IDs:", tokens["input_ids"])





In [ ]:
# check PyTorch and CUDA
import torch

print("Torch version:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU RAM GB:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    device = torch.device("cuda") # store CUDA for model.to
else:
    device = torch.device("cpu")

print("Selected device:", device)

In [ ]:
# model load parameter

from transformers import EsmModel

model = EsmModel.from_pretrained(model_name)
model.eval()
model = model.to(device)

print("model:", model_name)
print("hidden dim:", model.config.hidden_size)
print("layers:", model.config.num_hidden_layers)
print("device:", next(model.parameters()).device)


In [ ]:
# compute embeddings

inputs = {key: value.to(device) for key, value in tokens.items()}

# run esm
with torch.inference_mode():
    outputs = model(
        **inputs,
        output_hidden_states=True,
        return_dict=True
    )

# extract last layer
last_hidden_states = outputs.hidden_states[-1]

# remove start-token: 0 = <cls>, end-token: 2 = <eos>
residue_embeddings = last_hidden_states[0, 1:-1, :]


print("hidden-states:", len(outputs.hidden_states))
print("residue embeddings matrix:", residue_embeddings.shape)

# ?? hidden-states = 34
# ESM-2 returns 34 hidden-state representations!
# hidden_states[0]: initial token embeddings, we do not need
# hidden_states[1:34]: representations from layers 1 to 33 !!!

#print("hidden-states used:", outputs.hidden_states[1:34])
print("hidden-states used:", len(outputs.hidden_states[1:34]))


In [ ]:
# check all BAliBASE
# filtering BAliBASE data max_length <= 1022, paper section 2.5 

input_dir = Path("datasets/BAliBASE/inputs")     ## check your path!!
fasta_files = sorted(input_dir.glob("*.fasta"))
#print(fasta_files)

print("#BAliBASE files:", len(fasta_files))


# files filtered
max_lengths = {}

for fasta_path in fasta_files:
    lengths = []
    current_length = 0

    with open(fasta_path, "r") as f:
        for line in f:
            line = line.strip()

            if line.startswith(">"):
                if current_length > 0:
                    lengths.append(current_length)
                current_length = 0
            else:
                current_length += len(line)

    if current_length > 0:
        lengths.append(current_length)

    max_lengths[fasta_path.stem] = max(lengths)

#  length restriction
filter_true = {
    name: length
    for name, length in max_lengths.items()
    if length <= 1022
}

filter_false = {
    name: length
    for name, length in max_lengths.items()
    if length > 1022
}


print("filter true datasets:", len(filter_true))
print("filter false datasets:", len(filter_false))



In [ ]:
# 168 vs 172 ?? files with gaps??

files_with_gaps = []

for fasta_path in fasta_files:
    with open(fasta_path, "r") as f:
        for line in f:
            line = line.strip()

            if not line.startswith(">") and ("-" in line or "." in line):
                files_with_gaps.append(fasta_path.name)
                break

print("files with gaps:", len(files_with_gaps))
print("first gapped files", files_with_gaps[:10])




In [ ]:
# no gaps, difference 168 reported to 172 has to be investigated!